# 4 · Ranking from sparse pairwise comparisons

Companion to **Section 6**. Alternatives are nodes, available judgements
$S_{ij}\approx w_i/w_j$ are edges, and with $b_h=\ln S_{ij}$ the logarithmic
least-squares estimator is the master problem with $W=I$. Its distributed
solver is consensus plus a constant bias (Algorithm 1).

In [ ]:
import sys
sys.path.insert(0, "../code")          # the scripts that generate the paper's figures
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
np.set_printoptions(precision=4, suppress=True)


def draw(ax, P, edges, nodes_kw=None, title=None):
    """Draw a graph with node positions P (n x 2) and an edge list."""
    ax.add_collection(LineCollection([[P[i], P[j]] for i, j in edges],
                                     colors="0.75", linewidths=0.6, zorder=1))
    ax.scatter(P[:, 0], P[:, 1], s=14, zorder=2, **(nodes_kw or {}))
    ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)

In [ ]:
from scipy.stats import kendalltau
from common import graph_ahp, incidence, spectra
G, A, d, L = graph_ahp()
B, edges = incidence(G)
lam, *_ = spectra(L, d)
n, m = B.shape
print(f"n = {n}, m = {m}, degrees {d.min():.0f}..{d.max():.0f}, lambda_2 = {lam[1]:.4f}")

In [ ]:
rng = np.random.default_rng(1)
w_true = np.exp(rng.uniform(0, np.log(9), n))     # true utilities
sigma = 0.3
b = np.array([np.log(w_true[i] / w_true[j]) for i, j in edges]) + rng.normal(0, sigma, m)

# centralized solution of L r = B b (zero-mean gauge)
r_star = np.linalg.pinv(L) @ (B @ b)
w_hat = np.exp(r_star)
print("Kendall tau:", kendalltau(w_true, w_hat).statistic)
print("inconsistency (optimal residual):", np.linalg.norm(B.T @ r_star - b))

## Algorithm 1: each agent only talks to the alternatives it was compared with

$r_i(k+1)=r_i(k)+\theta\sum_{j\in\mathcal N_i}(r_j(k)-r_i(k))+\theta s_i$, with $s_i=\sum_{j\in\mathcal N_i}\ln S_{ij}$.

In [ ]:
s = B @ b
theta = 1 / d.max()
rho = max(abs(1 - theta * lam[1]), abs(1 - theta * lam[-1]))
r = rng.normal(0, 1, n)
r_inf = r_star + r.mean()                 # the flow preserves the mean of r(0)
err = []
for k in range(300):
    err.append(np.linalg.norm(r - r_inf))
    r = r + theta * (-(L @ r) + s)          # = consensus step + theta * s
plt.semilogy(err, label="error"); plt.semilogy(err[0] * rho ** np.arange(300), "--", label="rho^k")
plt.legend(); plt.xlabel("k"); plt.show()
print(f"theta = {theta:.4f}, rho = {rho:.4f}")

*Consensus is sparse AHP with no information on the edges*: set `b = np.zeros(m)`
above and the iteration converges to the average of `r(0)`.

## Ranking accuracy against the noise level

In [ ]:
Lp = np.linalg.pinv(L)
sigmas, taus = np.linspace(0, 1, 6), []
for sg in sigmas:
    t = []
    for _ in range(100):
        w = np.exp(rng.uniform(0, np.log(9), n))
        bb = B.T @ np.log(w) + rng.normal(0, sg, m)
        t.append(kendalltau(w, np.exp(Lp @ (B @ bb))).statistic)
    taus.append(np.mean(t))
plt.plot(sigmas, taus, "o-"); plt.xlabel("sigma"); plt.ylabel("Kendall tau"); plt.show()

*Try it.* Exercise 8 adds several decision makers per pair ($W\neq I$);
Exercise 9 shows that the variance of $\hat r_i-\hat r_j$ is
$\sigma^2$ times the effective resistance between $i$ and $j$.